# Cascade Mask R-CNN — inference on RTS test chips

Point `MODEL_PATH` at a checkpoint, run top to bottom, look at the masks.

**What this does**

1. Loads a trained checkpoint with the *same* config the training run used.
2. Runs it over a handful of `test/` chips and draws the predicted masks.
3. Optionally does the same on `val/`, where there **is** ground truth, so you can
   see truth and prediction side by side and score the model honestly.
4. Optionally re-exports a full `submission.json`.

**Requirements.** detectron2 — which does not build easily on Windows. On Kaggle
(GPU on, internet on for the first run) cell 2 installs it for you. Locally, expect
to run this under WSL or Linux; a CPU-only box works fine at this scale, just slower
(a second or two per chip).

In [1]:
# ---------------------------------------------------------------------------
# CONFIG - everything worth changing lives here.
# ---------------------------------------------------------------------------
# Checkpoint to run. None -> auto-detect (prefers model_best.pth from early stopping,
# then model_0013999.pth, then model_final.pth).
# On val, model_0013999 scored segm AP 52.13 against 52.05 for model_final: the last
# 1000 iterations very slightly overfit, so the earlier checkpoint is the better pick.
MODEL_PATH = r'C:\Users\moham\Downloads\cascade rcnn\output\model_0013999.pth'      # e.g. r"C:/Users/moham/Downloads/cascade rcnn/output/model_0013999.pth"
DATA_ROOT  = r'C:\Users\moham\Downloads\2026_geoai_arctic_challenge_v1.0\data_png'      # None -> auto-detect data_png/

# --- which test chips to run ------------------------------------------------
N_IMAGES  = 6          # how many to sample
IMAGE_IDS = None       # e.g. [49, 97, 113] to pin specific ones; overrides N_IMAGES
SEED      = 42         # which random sample you get

# --- display ----------------------------------------------------------------
SCORE_THRESH = 0.5     # masks drawn at or above this score
TOP_K        = 10      # the challenge scorer uses maxDets=10
COLS         = 3

# --- must match the training run, do not change these to "tune" anything ----
CONFIG_YAML          = "Misc/cascade_mask_rcnn_R_50_FPN_3x.yaml"
SCORE_THRESH_TEST    = 0.05   # what the model keeps internally
DETECTIONS_PER_IMAGE = 20

# --- optional extras --------------------------------------------------------
RUN_VAL_CHECK    = True    # visual GT vs prediction on val chips (val has labels)
VAL_N            = 4
SCORE_FULL_VAL   = False   # official AP over all 114 val chips (minutes on CPU)
EXPORT_FULL_TEST = False   # re-run all 138 test chips -> submission.json

print(f"sampling {len(IMAGE_IDS) if IMAGE_IDS else N_IMAGES} test chips, "
      f"drawing masks at score >= {SCORE_THRESH}")

sampling 6 test chips, drawing masks at score >= 0.5


## 1 · Environment

In [ ]:
import importlib
import subprocess
import sys

import numpy as np
import torch

print("torch :", torch.__version__, "| cuda:", torch.version.cuda,
      "| gpu:", torch.cuda.is_available())
print("numpy :", np.__version__)

if int(np.__version__.split(".")[0]) >= 2:
    print("WARNING: numpy 2.x - detectron2's extensions need numpy<2.")
    print("         Run: pip install 'numpy<2'  then RESTART the kernel.")

try:
    import detectron2
    import detectron2._C                      # the compiled extension - the real test
    print("detectron2:", detectron2.__version__, "(already present)")
except ImportError:
    print("building detectron2 from source, ~10 min ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/facebookresearch/detectron2.git"])
    importlib.invalidate_caches()
    import detectron2
    import detectron2._C
    print("detectron2:", detectron2.__version__, "(built)")

## 2 · Locate the data and the checkpoint

In [ ]:
import csv
import json
import os
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
from pycocotools import mask as mask_utils
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval

from detectron2 import model_zoo
from detectron2.checkpoint import DetectionCheckpointer
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_test_loader)
from detectron2.data.datasets import register_coco_instances
from detectron2.modeling import build_model
from detectron2.utils.logger import setup_logger

setup_logger()
random.seed(SEED)
np.random.seed(SEED)

# Bounded search roots - deliberately not a recursive walk of your home directory.
SEARCH_ROOTS = [Path("/kaggle/input"), Path("/kaggle/working"), Path("."), Path(".."),
                Path.home() / "Downloads"]


def find_data_root(explicit=None) -> Path:
    """Locate data_png/ by its annotations file."""
    if explicit:
        p = Path(explicit)
        if not (p / "annotations" / "instances_val_fold.json").exists():
            raise FileNotFoundError(f"{p} does not look like data_png/")
        return p
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for pat in ("annotations/instances_train_fold.json",
                    "*/annotations/instances_train_fold.json",
                    "*/*/annotations/instances_train_fold.json"):
            for hit in sorted(root.glob(pat)):
                return hit.parent.parent
    raise FileNotFoundError(
        "data_png not found. Set DATA_ROOT, attach it as a Kaggle Dataset, or run "
        "prepare_png_dataset.py locally."
    )


def find_checkpoint(explicit=None) -> Path:
    """Find a trained checkpoint, preferring the one that scored best on val."""
    if explicit:
        p = Path(explicit)
        if not p.exists():
            raise FileNotFoundError(f"no checkpoint at {p}")
        return p

    hits = []
    for root in SEARCH_ROOTS:
        if not root.exists():
            continue
        for pat in ("output/model_*.pth", "*/output/model_*.pth",
                    "*/*/output/model_*.pth", "model_*.pth"):
            hits += list(root.glob(pat))
    hits = sorted({h.resolve() for h in hits})
    if not hits:
        raise FileNotFoundError(
            "no model_*.pth found. Set MODEL_PATH to your checkpoint explicitly."
        )

    by_name = {h.name: h for h in hits}
    for preferred in ("model_best.pth", "model_0013999.pth", "model_final.pth"):
        if preferred in by_name:
            return by_name[preferred]
    return sorted(hits, key=lambda p: p.name)[-1]


DATA = find_data_root(DATA_ROOT)
CKPT = find_checkpoint(MODEL_PATH)
print("data root :", DATA)
print("checkpoint:", CKPT, f"({CKPT.stat().st_size / 1e6:.0f} MB)")

## 3 · Build the model

Every setting here has to match training, or the checkpoint loads into the wrong shape
of network. The three that actually matter: `NUM_CLASSES = 1` (it propagates to all
three cascade heads), `MASK_FORMAT = "bitmask"` because the labels are RLE rather than
polygons, and `INPUT.FORMAT` left at its default `BGR` — which is why chips are read
with `cv2.imread` and *not* flipped to RGB before they reach the model.

Watch the load log below. A `Skipping`/`shape mismatch` line on any `roi_heads` weight
means the config and the checkpoint disagree, and the predictions will be garbage.

In [ ]:
def build_inference_cfg(weights):
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(CONFIG_YAML))
    cfg.MODEL.WEIGHTS = str(weights)

    cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1        # propagates to all three cascade heads
    cfg.INPUT.MASK_FORMAT = "bitmask"          # labels are RLE, not polygons

    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = SCORE_THRESH_TEST
    cfg.TEST.DETECTIONS_PER_IMAGE = DETECTIONS_PER_IMAGE
    cfg.MODEL.DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    cfg.DATALOADER.NUM_WORKERS = 0             # 0 keeps notebooks and Windows happy
    return cfg


cfg_i = build_inference_cfg(CKPT)
model = build_model(cfg_i)
DetectionCheckpointer(model).load(cfg_i.MODEL.WEIGHTS)
model.eval()

print("\ndevice        :", cfg_i.MODEL.DEVICE)
print("roi heads     :", cfg_i.MODEL.ROI_HEADS.NAME,
      "| classes:", cfg_i.MODEL.ROI_HEADS.NUM_CLASSES)
print("input format  :", cfg_i.INPUT.FORMAT, "| pixel mean:", cfg_i.MODEL.PIXEL_MEAN)
print("min_size_test :", cfg_i.INPUT.MIN_SIZE_TEST,
      "-> these ~290x150 chips get upsampled ~2.8x")

## 4 · Inference helpers

In [ ]:
def encode_binary_mask(mask):
    """Encode an H x W binary mask as compressed COCO RLE (JSON-safe)."""
    rle = mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))
    rle["counts"] = rle["counts"].decode("utf-8")
    return rle


def rle_to_mask(rle):
    """Decode a COCO RLE back to an H x W bool array."""
    r = dict(rle)
    if isinstance(r["counts"], str):
        r["counts"] = r["counts"].encode("utf-8")
    return mask_utils.decode(r).astype(bool)


def predict(dataset, top_k=TOP_K):
    """Run the model over a list of dataset dicts -> submission-format records.

    Same code path that produced submission.json, so results here reproduce it
    exactly rather than approximating it.
    """
    loader = build_detection_test_loader(
        dataset, mapper=DatasetMapper(cfg_i, is_train=False))
    results = []
    with torch.no_grad():
        for batch in loader:
            for inp, out in zip(batch, model(batch)):
                inst = out["instances"].to("cpu")
                order = inst.scores.argsort(descending=True)[:top_k]
                for i in order.tolist():
                    results.append({
                        "image_id": int(inp["image_id"]),
                        "category_id": 1,
                        "segmentation": encode_binary_mask(inst.pred_masks[i].numpy()),
                        "score": float(inst.scores[i]),
                    })
    return results


GT_C, PR_C = (0.10, 0.95, 0.55), (1.0, 0.36, 0.10)   # green = truth, orange = prediction


def overlay(ax, img, insts, color, show_scores=True):
    """Draw instance masks as outline + tint on one axis."""
    ax.imshow(img)
    h, w = img.shape[:2]
    union = np.zeros((h, w), bool)
    for it in insts:
        m = rle_to_mask(it["segmentation"])
        union |= m
        ax.contour(m, levels=[0.5], colors=[color], linewidths=1.6)
        ys, xs = np.where(m)
        if len(xs) and show_scores and "score" in it:
            ax.text(xs.min(), max(ys.min() - 3, 8), f"{it['score']:.2f}",
                    color="white", fontsize=6.5,
                    bbox=dict(fc=color, ec="none", pad=0.9, alpha=0.85))
    tint = np.zeros((h, w, 4))
    tint[..., :3] = color
    tint[..., 3] = union * 0.28
    ax.imshow(tint)
    ax.set_xticks([])
    ax.set_yticks([])
    return union.sum() / (h * w)


def load_rgb(path):
    """Read a chip for display. The model gets BGR; matplotlib wants RGB."""
    bgr = cv2.imread(str(path))
    if bgr is None:
        raise FileNotFoundError(path)
    return bgr[:, :, ::-1]

## 5 · Run on test chips

In [2]:
with open(DATA / "test_manifest.csv", newline="") as f:
    rows = list(csv.DictReader(f))

test_dicts = [{
    "file_name": str(DATA / "test" / f"{r['public_id']}.png"),
    "image_id": int(r["image_id"]),
    "height": int(r["height"]),
    "width": int(r["width"]),
} for r in rows]
print(f"{len(test_dicts)} test chips available")

if IMAGE_IDS:
    chosen = [d for d in test_dicts if d["image_id"] in set(IMAGE_IDS)]
    missing = set(IMAGE_IDS) - {d["image_id"] for d in chosen}
    if missing:
        print(f"warning: no such image_id(s): {sorted(missing)}")
else:
    chosen = random.Random(SEED).sample(test_dicts, min(N_IMAGES, len(test_dicts)))
chosen.sort(key=lambda d: d["image_id"])

preds = predict(chosen)
per_image = {}
for p in preds:
    per_image.setdefault(p["image_id"], []).append(p)

print(f"ran {len(chosen)} chips -> {len(preds)} raw detections "
      f"(score >= {SCORE_THRESH_TEST})")
print(f"{sum(1 for p in preds if p['score'] >= SCORE_THRESH)} of them at "
      f"score >= {SCORE_THRESH}")

NameError: name 'DATA' is not defined

In [ ]:
n = len(chosen)
cols = min(COLS, n)
rows_n = (n + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(5.6 * cols, 3.4 * rows_n),
                         squeeze=False)

for ax, d in zip(axes.ravel(), chosen):
    img = load_rgb(d["file_name"])
    keep = [p for p in sorted(per_image.get(d["image_id"], []),
                              key=lambda p: -p["score"])
            if p["score"] >= SCORE_THRESH]
    frac = overlay(ax, img, keep, PR_C)
    ax.set_title(f"{Path(d['file_name']).name}  (id {d['image_id']})\n"
                 f"{len(keep)} inst - {100 * frac:.1f}% of frame", fontsize=9)

for ax in axes.ravel()[n:]:
    ax.axis("off")

fig.suptitle(f"TEST predictions - {CKPT.name} @ score >= {SCORE_THRESH}",
             fontsize=13, fontweight="bold")
fig.tight_layout()
plt.show()

### Threshold sensitivity

There is no ground truth for `test/`, so the honest thing to look at here is how much
the output moves as the threshold moves. Masks that appear and vanish across this range
are the model telling you its confidence is poorly calibrated on that chip.

In [3]:
THRESHOLDS = [0.05, 0.3, 0.5, 0.9]
probe = chosen[0]
img = load_rgb(probe["file_name"])
dets = sorted(per_image.get(probe["image_id"], []), key=lambda p: -p["score"])

fig, axes = plt.subplots(1, len(THRESHOLDS), figsize=(4.6 * len(THRESHOLDS), 3.4))
for ax, t in zip(axes, THRESHOLDS):
    keep = [p for p in dets if p["score"] >= t]
    overlay(ax, img, keep, PR_C)
    ax.set_title(f"score >= {t}  ->  {len(keep)} inst", fontsize=10)
fig.suptitle(f"{Path(probe['file_name']).name} - threshold sensitivity",
             fontsize=12, fontweight="bold")
fig.tight_layout()
plt.show()

print("all scores on this chip:", [round(p["score"], 3) for p in dets])

NameError: name 'chosen' is not defined

## 6 · Sanity check on val, where there *is* ground truth

`competition_release/test/` ships images only — no labels. So "is this model any good"
can only be answered on the val fold. Green is truth, orange is the prediction.

In [ ]:
if RUN_VAL_CHECK:
    if "rts_val" in DatasetCatalog.list():
        DatasetCatalog.remove("rts_val")
        MetadataCatalog.remove("rts_val")
    register_coco_instances("rts_val", {},
                            str(DATA / "annotations" / "instances_val_fold.json"),
                            str(DATA / "train"))
    MetadataCatalog.get("rts_val").thing_classes = ["rts"]
    val_dicts = DatasetCatalog.get("rts_val")

    val_pick = random.Random(SEED).sample(val_dicts, min(VAL_N, len(val_dicts)))
    val_pick.sort(key=lambda d: d["image_id"])
    val_preds = predict(val_pick)

    vper = {}
    for p in val_preds:
        vper.setdefault(p["image_id"], []).append(p)

    k = len(val_pick)
    fig, axes = plt.subplots(2, k, figsize=(5.0 * k, 7.0), squeeze=False)
    for c, d in enumerate(val_pick):
        img = load_rgb(d["file_name"])
        gt = [{"segmentation": a["segmentation"]} for a in d["annotations"]]
        overlay(axes[0, c], img, gt, GT_C)
        axes[0, c].set_title(f"{Path(d['file_name']).name} - GT ({len(gt)})", fontsize=9)

        keep = [p for p in vper.get(d["image_id"], []) if p["score"] >= SCORE_THRESH]
        overlay(axes[1, c], img, keep, PR_C)
        axes[1, c].set_title(f"pred @score >= {SCORE_THRESH} ({len(keep)})", fontsize=9)

    fig.suptitle("VAL - ground truth (top, green) vs prediction (bottom, orange)",
                 fontsize=13, fontweight="bold")
    fig.tight_layout()
    plt.show()
else:
    print("RUN_VAL_CHECK is False - skipping")

### Official score

The challenge does **not** use stock COCO settings: it scores at `maxDets=[1, 5, 10]`
with its own small/medium/large area bins. The AP printed by `COCOEvaluator` during
training is therefore *not* the ranking metric — compare runs on the numbers below.

Alongside the ranking AP this reports AP50, AP75 and AP small/medium/large on the official
bins, plus two mask-IoU numbers: **matched IoU** (mean IoU of prediction/GT pairs matched at
IoU ≥ 0.5, top 10 per image) and **foreground IoU** (pixel IoU of all predictions scoring
≥ 0.5 against all ground truth, pooled over the fold).

In [ ]:
OFFICIAL_MAXDETS = [1, 5, 10]
OFFICIAL_AREA_RNG = [[0, 1e10], [0, 300], [300, 2000], [2000, 1e10]]
OFFICIAL_AREA_LBL = ["all", "small", "medium", "large"]


def _ap(ev, area="all", iou_thr=None, max_det=10):
    """Mean precision for one row of the challenge's summary table."""
    prec = ev.eval["precision"]
    if iou_thr is not None:
        prec = prec[np.isclose(ev.params.iouThrs, iou_thr)]
    prec = prec[:, :, :, ev.params.areaRngLbl.index(area), ev.params.maxDets.index(max_det)]
    valid = prec[prec > -1]
    return float(valid.mean()) if valid.size else -1.0


def iou_metrics(coco_gt, predictions, max_det=10, match_thr=0.5, fg_score=0.5):
    """Two mask-IoU numbers that AP does not show directly.

    matched_iou    - mean IoU of prediction/GT pairs, greedily matched by score
                     (top `max_det` per image, IoU >= `match_thr`), as COCOeval does.
    foreground_iou - pixel IoU of the union of predictions scoring >= `fg_score`
                     against the union of GT masks, summed over all images.
    """
    by_img = {}
    for p in predictions:
        by_img.setdefault(p["image_id"], []).append(p)

    matched, inter, union = [], 0, 0
    for img_id in coco_gt.getImgIds():
        info = coco_gt.imgs[img_id]
        h, w = info["height"], info["width"]
        gts = [coco_gt.annToRLE(a) for a in coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=img_id))]
        dts = sorted(by_img.get(img_id, []), key=lambda p: -p["score"])[:max_det]

        if gts and dts:
            ious = mask_utils.iou([d["segmentation"] for d in dts], gts, [0] * len(gts))
            taken = set()
            for row in np.atleast_2d(ious):
                best, best_j = match_thr, -1
                for j, v in enumerate(row):
                    if j not in taken and v >= best:
                        best, best_j = v, j
                if best_j >= 0:
                    taken.add(best_j)
                    matched.append(best)

        gt_fg = np.zeros((h, w), bool)
        for r in gts:
            gt_fg |= mask_utils.decode(r).astype(bool)
        dt_fg = np.zeros((h, w), bool)
        for d in dts:
            if d["score"] >= fg_score:
                dt_fg |= mask_utils.decode(d["segmentation"]).astype(bool)
        inter += int((gt_fg & dt_fg).sum())
        union += int((gt_fg | dt_fg).sum())

    return {
        "matched_iou": float(np.mean(matched)) if matched else 0.0,
        "n_matched": len(matched),
        "foreground_iou": inter / union if union else 0.0,
    }


def score_official(gt_json_path, predictions):
    """COCO segm AP using the challenge's maxDets and area ranges, plus mask IoU."""
    if not predictions:
        print("no predictions - nothing to score")
        return None
    coco_gt = COCO(str(gt_json_path))
    # copies: loadRes writes bbox/area into each dict, which corrupts a second scoring
    coco_dt = coco_gt.loadRes([dict(p) for p in predictions])
    ev = COCOeval(coco_gt, coco_dt, "segm")
    ev.params.maxDets = OFFICIAL_MAXDETS
    ev.params.areaRng = OFFICIAL_AREA_RNG
    ev.params.areaRngLbl = OFFICIAL_AREA_LBL
    ev.evaluate()
    ev.accumulate()

    metrics = {
        "AP":        _ap(ev),
        "AP50":      _ap(ev, iou_thr=0.50),
        "AP75":      _ap(ev, iou_thr=0.75),
        "AP_small":  _ap(ev, area="small"),
        "AP_medium": _ap(ev, area="medium"),
        "AP_large":  _ap(ev, area="large"),
    }
    n_gt = len(coco_gt.getAnnIds())
    metrics.update(iou_metrics(coco_gt, predictions))

    print(f"AP        @[IoU=0.50:0.95 | area=all    | maxDets=10] = {metrics['AP']:.4f}   <- ranking metric")
    print(f"AP50      @[IoU=0.50      | area=all    | maxDets=10] = {metrics['AP50']:.4f}")
    print(f"AP75      @[IoU=0.75      | area=all    | maxDets=10] = {metrics['AP75']:.4f}")
    print(f"AP_small  @[IoU=0.50:0.95 | area=small  | maxDets=10] = {metrics['AP_small']:.4f}")
    print(f"AP_medium @[IoU=0.50:0.95 | area=medium | maxDets=10] = {metrics['AP_medium']:.4f}")
    print(f"AP_large  @[IoU=0.50:0.95 | area=large  | maxDets=10] = {metrics['AP_large']:.4f}")
    print(f"matched mask IoU (IoU>=0.5)                        = {metrics['matched_iou']:.4f}"
          f"   ({metrics['n_matched']}/{n_gt} GT instances matched)")
    print(f"foreground IoU   (score>=0.5, all images)          = {metrics['foreground_iou']:.4f}")
    return metrics



if SCORE_FULL_VAL:
    if "rts_val" not in DatasetCatalog.list():        # registered in section 6 only if RUN_VAL_CHECK
        register_coco_instances("rts_val", {},
                                str(DATA / "annotations" / "instances_val_fold.json"),
                                str(DATA / "train"))
        MetadataCatalog.get("rts_val").thing_classes = ["rts"]
    val_all_dicts = DatasetCatalog.get("rts_val")
    all_val = predict(val_all_dicts)
    print(f"{len(all_val)} predictions over {len(val_all_dicts)} val images\n")
    val_metrics = score_official(DATA / "annotations" / "instances_val_fold.json", all_val)
else:
    print("SCORE_FULL_VAL is False - set it True to score all 114 val chips "
          "(a few minutes on CPU)")

## 7 · Optional: re-export a full submission

In [ ]:
if EXPORT_FULL_TEST:
    test_preds = predict(test_dicts)
    out_path = Path("submission.json")
    out_path.write_text(json.dumps(test_preds))
    print(f"wrote {out_path.resolve()}  ({len(test_preds)} predictions)")

    # Validate before trusting it - a malformed submission fails quietly at scoring.
    sizes = {int(r["image_id"]): (int(r["height"]), int(r["width"])) for r in rows}
    seen = {}
    for i, p in enumerate(test_preds):
        assert set(p) == {"image_id", "category_id", "segmentation", "score"}, i
        assert p["image_id"] in sizes, f"unknown image_id {p['image_id']}"
        assert p["category_id"] == 1, "category_id must be 1"
        assert 0.0 <= p["score"] <= 1.0 and np.isfinite(p["score"]), "bad score"
        assert p["segmentation"]["size"] == list(sizes[p["image_id"]]), "RLE size mismatch"
        seen[p["image_id"]] = seen.get(p["image_id"], 0) + 1

    crowded = {k: v for k, v in seen.items() if v > 10}
    print(f"validation=ok  images_with_predictions={len(seen)}/{len(rows)}")
    if crowded:
        print(f"note: {len(crowded)} images exceed maxDets=10")
else:
    print("EXPORT_FULL_TEST is False - skipping")